# Assignment 4 — Reweighing fairness evaluation

Kamiran & Calders (2012), *Data preprocessing techniques for classification without discrimination*, §5.2 (Algorithm 3). Implementation: IBM AIF360 `Reweighing`.

**Design:** Adult (sex), German Credit (age >25), COMPAS (race); 80/20 joint-label/group stratification; seed 42; Logistic Regression (`C=1`, `lbfgs`, `max_iter=2000`), threshold 0.5. Protected attribute is used for weighting and evaluation, not as a model predictor. Training-only transformations; unmodified test data. Other non-target demographics can remain among predictors.

Fairness: `DP_Diff = P(positive | unprivileged) − P(positive | privileged)` (target 0); `DIR = P(positive | unprivileged) / P(positive | privileged)` (target 1); 5-neighbor Consistency (target 1, includes self as in AIF360). `1` denotes the favorable outcome in all datasets.

Original-label metrics cover the entire dataset and the held-out test subset; the effect of mitigation is assessed by comparing baseline and reweighed predictions on the same test subset.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from aif360.datasets import BinaryLabelDataset
from aif360.algorithms.preprocessing.reweighing import Reweighing

SEED = 42
TEST_SIZE = 0.20
K = 5  # Neighbors for consistency (including self, as in AIF360)

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "adult" / "adult.data").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open Assignment4 in VS Code and run this notebook there.")

DATA = {
    "Adult train": PROJECT_ROOT / "adult" / "adult.data",
    "Adult test": PROJECT_ROOT / "adult" / "adult.test",
    "German": PROJECT_ROOT / "statlog+german+credit+data" / "german.data",
    "COMPAS": PROJECT_ROOT / "compas-scores-two-years.csv",
}
missing = [str(p) for p in DATA.values() if not p.is_file()]
if missing:
    raise FileNotFoundError("Missing data files:\n" + "\n".join(missing))

RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)
print("Data files found. Python kernel is ready.")

Data files found. Python kernel is ready.


d:\Development\CondaEnvs\ads\Lib\site-packages\inFairness\utils\ndcg.py:37: FutureWarning: We've integrated functorch into PyTorch. As the final step of the integration, `functorch.vmap` is deprecated as of PyTorch 2.0 and will be deleted in a future version of PyTorch >= 2.3. Please use `torch.vmap` instead; see the PyTorch 2.0 release notes and/or the `torch.func` migration guide for more details https://pytorch.org/docs/main/func.migrating.html
  vect_normalized_discounted_cumulative_gain = vmap(
d:\Development\CondaEnvs\ads\Lib\site-packages\inFairness\utils\ndcg.py:48: FutureWarning: We've integrated functorch into PyTorch. As the final step of the integration, `functorch.vmap` is deprecated as of PyTorch 2.0 and will be deleted in a future version of PyTorch >= 2.3. Please use `torch.vmap` instead; see the PyTorch 2.0 release notes and/or the `torch.func` migration guide for more details https://pytorch.org/docs/main/func.migrating.html
  monte_carlo_vect_ndcg = vmap(vect_normali

## 1. Load and prepare datasets

- Adult: income >$50K; female = 0, male = 1. Missing records removed; age and education binned.
- German Credit: good credit; age ≤25 = 0, age >25 = 1. Credit history, savings and employment grouped.
- COMPAS: no two-year recidivism; African-American = 0, Caucasian = 1. ProPublica/AIF360 screening filters applied.

In [2]:
ADULT_COLUMNS = [
    "age", "workclass", "fnlwgt", "education", "education-num",
    "marital-status", "occupation", "relationship", "race", "sex",
    "capital-gain", "capital-loss", "hours-per-week", "native-country",
    "income-per-year",
]
GERMAN_COLUMNS = [
    "status", "month", "credit_history", "purpose", "credit_amount",
    "savings", "employment", "investment_as_income_percentage", "personal_status",
    "other_debtors", "residence_since", "property", "age", "installment_plans",
    "housing", "number_of_credits", "skill_level", "people_liable_for",
    "telephone", "foreign_worker", "credit",
]


def load_adult(data_root):
    folder = data_root / "adult"
    common = dict(header=None, names=ADULT_COLUMNS, skipinitialspace=True,
                  na_values=["?", " ?"], engine="python")
    train_raw = pd.read_csv(folder / "adult.data", **common)
    test_raw = pd.read_csv(folder / "adult.test", skiprows=1, **common)
    df = pd.concat([train_raw, test_raw], ignore_index=True)
    df["income-per-year"] = df["income-per-year"].astype(str).str.strip().str.rstrip(".")
    df = df.dropna().copy()
    df = df[df["income-per-year"].isin([">50K", "<=50K"]) &
            df["sex"].isin(["Male", "Female"])].copy()
    df["age_decade"] = (df["age"].astype(int) // 10 * 10).clip(upper=70).astype(str)
    edu = df["education-num"].astype(int)
    df["education_years"] = np.where(
        edu <= 5, "<6", np.where(edu >= 13, ">12", edu.astype(str))
    )
    df["race_white"] = (df["race"] == "White").astype(float)
    X = df[["age_decade", "education_years", "race_white"]].reset_index(drop=True)
    y = (df["income-per-year"] == ">50K").astype(int).to_numpy()
    protected = (df["sex"] == "Male").astype(int).to_numpy()
    return X, y, protected, ["age_decade", "education_years"], ["race_white"]


def load_german(data_root):
    df = pd.read_csv(data_root / "statlog+german+credit+data" / "german.data", sep=r"\s+",
                     header=None, names=GERMAN_COLUMNS).copy()
    df = df.dropna(subset=["credit", "age", "personal_status",
                           "credit_history", "savings", "employment"])
    df = df[df["credit"].isin([1, 2])].copy()
    df["credit_history_group"] = df["credit_history"].map({
        "A30": "None/Paid", "A31": "None/Paid", "A32": "None/Paid",
        "A33": "Delay", "A34": "Other"}).fillna("Unknown")
    df["savings_group"] = df["savings"].map({
        "A61": "<500", "A62": "<500", "A63": "500+", "A64": "500+",
        "A65": "Unknown/None"}).fillna("Unknown")
    df["employment_group"] = df["employment"].map({
        "A71": "Unemployed", "A72": "1-4 years", "A73": "1-4 years",
        "A74": "4+ years", "A75": "4+ years"}).fillna("Unknown")
    df["sex_male"] = df["personal_status"].isin(["A91", "A93", "A94"]).astype(float)
    X = df[["credit_history_group", "savings_group", "employment_group",
            "sex_male"]].reset_index(drop=True)
    y = (df["credit"] == 1).astype(int).to_numpy()
    protected = (df["age"] >= 26).astype(int).to_numpy()
    return X, y, protected, ["credit_history_group", "savings_group",
                              "employment_group"], ["sex_male"]


def load_compas(data_root):
    df = pd.read_csv(data_root / "compas-scores-two-years.csv",
                     low_memory=False)
    needed = ["days_b_screening_arrest", "is_recid", "c_charge_degree",
              "score_text", "race", "two_year_recid", "age_cat", "priors_count", "sex"]
    df = df.dropna(subset=needed).copy()
    df = df[
        df["days_b_screening_arrest"].between(-30, 30)
        & (df["is_recid"] != -1)
        & (df["c_charge_degree"] != "O")
        & (df["score_text"] != "N/A")
        & df["race"].isin(["African-American", "Caucasian"])
        & df["two_year_recid"].isin([0, 1])
    ].copy()
    df["priors_group"] = pd.cut(
        df["priors_count"], bins=[-np.inf, 0, 3, np.inf],
        labels=["0", "1 to 3", "More than 3"]
    ).astype(str)
    df["sex_female"] = (df["sex"] == "Female").astype(float)
    X = df[["age_cat", "priors_group", "c_charge_degree", "sex_female"]].reset_index(drop=True)
    y = (df["two_year_recid"] == 0).astype(int).to_numpy()  # 1 = NO recidivism
    protected = (df["race"] == "Caucasian").astype(int).to_numpy()
    return X, y, protected, ["age_cat", "priors_group", "c_charge_degree"], ["sex_female"]

DATASET_SPECS = {
    "Adult": (load_adult, "Sex: Female / Male"),
    "German": (load_german, "Age: <=25 / >25"),
    "COMPAS": (load_compas, "Race: African-American / Caucasian"),
}

## 2. Metrics and models
Consistency uses Euclidean distance on one-hot categorical and standardized numerical predictors. 


In [3]:
def feature_encoder(categorical, numeric):
    return ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical),
        ("num", StandardScaler(), numeric),
    ], sparse_threshold=0.0)


def neighbor_ids(X):
    return NearestNeighbors(n_neighbors=K, n_jobs=-1).fit(X).kneighbors(X)[1]


def evaluate(dataset, stage, y_values, group, neighbors, y_true=None):
    y_values = np.asarray(y_values, dtype=int)
    p0 = y_values[group == 0].mean()
    p1 = y_values[group == 1].mean()
    row = {
        "Dataset": dataset, "Stage": stage, "N": len(y_values),
        "DP_Diff": p0 - p1,
        "DIR": p0 / p1 if p1 > 0 else np.nan,
        "Consistency": 1 - np.abs(y_values - y_values[neighbors].mean(axis=1)).mean(),
        "Accuracy": np.nan, "F1": np.nan, "BalancedAccuracy": np.nan,
    }
    if y_true is not None:
        row.update({
            "Accuracy": accuracy_score(y_true, y_values),
            "F1": f1_score(y_true, y_values, zero_division=0),
            "BalancedAccuracy": balanced_accuracy_score(y_true, y_values),
        })
    return row

In [4]:
def run_experiment(name, loader):
    X, y, group, categorical, numeric = loader(PROJECT_ROOT)
    assert len(X) == len(y) == len(group)
    assert set(np.unique(group)) == {0, 1} and set(np.unique(y)) == {0, 1}
    assert (pd.crosstab(group, y).to_numpy() > 0).all(), "Empty group-label cell"

    # Original-label metrics before mitigation or classifier training.
    all_features = feature_encoder(categorical, numeric).fit_transform(X)
    original = evaluate(name, "Original labels (full)", y, group,
                        neighbor_ids(all_features))

    # Stratify by protected group AND true label.
    train_i, test_i = train_test_split(
        np.arange(len(y)), test_size=TEST_SIZE, random_state=SEED,
        stratify=2 * group + y,
    )
    y_train, y_test = y[train_i], y[test_i]
    group_train, group_test = group[train_i], group[test_i]

    # Fit all model feature transformations on TRAIN only.
    encoder = feature_encoder(categorical, numeric)
    X_train = encoder.fit_transform(X.iloc[train_i])
    X_test = encoder.transform(X.iloc[test_i])
    neighbors_test = neighbor_ids(X_test)
    original_test = evaluate(name, "Original labels (test)",
                             y_test, group_test, neighbors_test)

    model_params = dict(C=1.0, max_iter=2000, solver="lbfgs", random_state=SEED)
    baseline = LogisticRegression(**model_params).fit(X_train, y_train)
    baseline_pred = baseline.predict(X_test)
    baseline_metrics = evaluate(name, "Baseline predictions", baseline_pred,
                                group_test, neighbors_test, y_true=y_test)

    # Reweighing: AIF360 computes group/label weights on TRAIN only.
    aif_train = BinaryLabelDataset(
        df=pd.DataFrame({"protected": group_train.astype(float),
                         "label": y_train.astype(float)}),
        label_names=["label"], protected_attribute_names=["protected"],
        favorable_label=1.0, unfavorable_label=0.0,
    )
    rw = Reweighing(unprivileged_groups=[{"protected": 0.0}],
                    privileged_groups=[{"protected": 1.0}])
    rw.fit(aif_train)
    weights = rw.transform(aif_train).instance_weights
    assert np.isfinite(weights).all() and (weights > 0).all()
    weighted_dp = (
        np.average(y_train[group_train == 0], weights=weights[group_train == 0])
        - np.average(y_train[group_train == 1], weights=weights[group_train == 1])
    )
    assert abs(weighted_dp) < 1e-8

    mitigated = LogisticRegression(**model_params).fit(
        X_train, y_train, sample_weight=weights)
    rw_pred = mitigated.predict(X_test)
    rw_metrics = evaluate(name, "Reweighing predictions", rw_pred,
                          group_test, neighbors_test, y_true=y_test)

    info = {
        "Dataset": name, "N_total": len(y), "N_train": len(train_i),
        "N_test": len(test_i), "N_features": X_train.shape[1],
        "Train_weighted_DP_Diff": weighted_dp,
    }
    return [original, original_test, baseline_metrics, rw_metrics], info

## 3. Results

`Original labels (full)` satisfies the before-training requirement; `Original labels (test)` provides a like-for-like test reference. Compare **Baseline predictions** with **Reweighing predictions** to assess mitigation. For change columns, negative `Delta_abs_DP_Diff` and `Delta_abs_DIR_minus_1` mean closer group parity; positive `Delta_Consistency` and `Delta_Accuracy` mean higher respective scores. Weighted train DP_Diff should be approximately zero.

In [5]:
records, dataset_info = [], []
for name, (loader, protected_group) in DATASET_SPECS.items():
    rows, info = run_experiment(name, loader)
    records.extend(rows)
    dataset_info.append({**info, "Protected groups (0 / 1)": protected_group})

metrics_df = pd.DataFrame(records)
config_df = pd.DataFrame(dataset_info)

changes = []
for name in DATASET_SPECS:
    rows = metrics_df[metrics_df["Dataset"] == name].set_index("Stage")
    before, after = rows.loc["Baseline predictions"], rows.loc["Reweighing predictions"]
    changes.append({
        "Dataset": name,
        "Delta_abs_DP_Diff": abs(after["DP_Diff"]) - abs(before["DP_Diff"]),
        "Delta_abs_DIR_minus_1": abs(after["DIR"] - 1) - abs(before["DIR"] - 1),
        "Delta_Consistency": after["Consistency"] - before["Consistency"],
        "Delta_Accuracy": after["Accuracy"] - before["Accuracy"],
        "Delta_F1": after["F1"] - before["F1"],
    })
changes_df = pd.DataFrame(changes)

display(config_df.round(4))
display(metrics_df.round(4))
display(changes_df.round(4))

metrics_df.to_csv(RESULTS_DIR / "fairness_metrics.csv", index=False)
changes_df.to_csv(RESULTS_DIR / "fairness_changes.csv", index=False)
config_df.to_csv(RESULTS_DIR / "experiment_config.csv", index=False)
print("Results saved to:", RESULTS_DIR)

,Dataset,N_total,N_train,N_test,N_features,Train_weighted_DP_Diff,Protected groups (0 / 1)
0,Adult,45222,36177,9045,17,-0.0,Sex: Female / Male
1,German,1000,800,200,10,-0.0,Age: <=25 / >25
2,COMPAS,5278,4222,1056,9,0.0,Race: African-American / Caucasian


,Dataset,Stage,N,DP_Diff,DIR,Consistency,Accuracy,F1,BalancedAccuracy
0,Adult,Original labels (full),45222,-0.1989,0.3635,0.7117,NaN,NaN,NaN
1,Adult,Original labels (test),9045,-0.1988,0.3637,0.7060,NaN,NaN,NaN
2,Adult,Baseline predictions,9045,-0.0738,0.6367,1.0000,0.7837,0.4935,0.6635
3,Adult,Reweighing predictions,9045,-0.0678,0.6751,1.0000,0.7833,0.5013,0.6680
4,German,Original labels (full),1000,-0.1494,0.7948,0.6408,NaN,NaN,NaN
5,German,Original labels (test),200,-0.1494,0.7948,0.6400,NaN,NaN,NaN
6,German,Baseline predictions,200,0.0247,1.0253,0.9940,0.7000,0.8214,0.5095
7,German,Reweighing predictions,200,0.0000,1.0000,1.0000,0.7000,0.8235,0.5000
8,COMPAS,Original labels (full),5278,-0.1323,0.7828,0.5941,NaN,NaN,NaN
9,COMPAS,Original labels (test),1056,-0.1309,0.7847,0.5578,NaN,NaN,NaN


,Dataset,Delta_abs_DP_Diff,Delta_abs_DIR_minus_1,Delta_Consistency,Delta_Accuracy,Delta_F1
0,Adult,-0.0060,-0.0385,0.000,-0.0004,0.0077
1,German,-0.0247,-0.0253,0.006,0.0000,0.0021
2,COMPAS,0.0000,0.0000,0.000,0.0000,0.0000


Results saved to: e:\Benjamin\Documents\University\Utrecht University\Applied Data Science\Data Wrangling (INFOMDW)\Assignment4\results
